# ConvNeXtV2 Tiny Pretrained Runs

Purpose: collect the pretrained ConvNeXtV2 Tiny artifacts in one notebook. This includes the linear-probe artifacts and the currently suspicious fine-tune artifact.

The fine-tune row with 0.25 accuracy should be treated as a failed or suspect diagnostic run unless it is rerun and verified.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

master_path = ROOT / 'reports' / 'model_comparison' / 'model_results_master.csv'
summary_path = ROOT / 'reports' / 'model_comparison' / 'model_results_master.json'
if not master_path.exists() or not summary_path.exists():
    display(Markdown("**Model results summary not found. Rebuilding from local metric artifacts...**"))
    subprocess.run([sys.executable, '-m', 'src.evaluation.build_model_results_master'], cwd=ROOT, check=True)

master = pd.read_csv(master_path)
summary = json.loads(summary_path.read_text(encoding='utf-8'))

metric_cols = [
    'model_name', 'run_group', 'pretrained', 'seed', 'split',
    'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
    'best_epoch', 'epochs_trained', 'recommendation', 'notes'
]

loaded_rows = master[master['status'] == 'loaded'].copy()
missing_rows = master[master['status'] != 'loaded'].copy()
if missing_rows.empty:
    display(Markdown(f"**Artifact status:** all `{len(master)}` result rows loaded from local metrics."))
else:
    display(Markdown(
        f"**Artifact status:** `{len(loaded_rows)}` loaded, `{len(missing_rows)}` missing. "
        "Missing rows mean the ignored metric/checkpoint artifacts are not present locally. "
        "Run the reproduction commands in this notebook, then rerun `python -m src.evaluation.build_model_results_master`."
    ))
    display(missing_rows[['model_family', 'run_group', 'seed', 'status', 'metrics_file']].head(20))

def rebuild_master_results():
    """Rebuild reports/model_comparison from whatever metric artifacts exist locally."""
    subprocess.run([sys.executable, '-m', 'src.evaluation.build_model_results_master'], cwd=ROOT, check=True)
    return pd.read_csv(master_path)

**Artifact status:** all `18` result rows loaded from local metrics.

In [2]:
convnext_pretrained = master[(master['model_family'] == 'convnextv2_tiny') & (master['pretrained'] == True)].copy()
convnext_pretrained[metric_cols].sort_values(['run_group', 'seed'])

,model_name,run_group,pretrained,seed,split,accuracy,precision_macro,recall_macro,f1_macro,best_epoch,epochs_trained,recommendation,notes
14,ConvNeXtV2 Tiny pretrained fine-tune,convnextv2_pretrained_finetune_suspect,True,NaN,local tune split from pretrained artifact,1.000000,1.000000,1.000000,1.000000,1.0,NaN,failed/suspect run,Accuracy is 0.25 on a tiny/support-limited tun...
13,ConvNeXtV2 Tiny pretrained linear probe seed123,convnextv2_pretrained_linear_probe,True,123.0,local tune split from pretrained artifact,1.000000,1.000000,1.000000,1.000000,4.0,NaN,comparison only,Second pretrained ConvNeXt linear-probe artifact.
12,ConvNeXtV2 Tiny pretrained linear probe,convnextv2_pretrained_linear_probe,True,NaN,local tune split from pretrained artifact,0.998214,0.998227,0.998214,0.998214,6.0,NaN,comparison only,Pretrained ConvNeXt linear-probe artifact from...


In [3]:
convnext_pretrained.groupby('run_group').agg(
    runs=('accuracy', 'count'),
    mean_accuracy=('accuracy', 'mean'),
    min_accuracy=('accuracy', 'min'),
    max_accuracy=('accuracy', 'max'),
    mean_f1=('f1_macro', 'mean'),
).reset_index()

,run_group,runs,mean_accuracy,min_accuracy,max_accuracy,mean_f1
0,convnextv2_pretrained_finetune_suspect,1,1.000000,1.000000,1.0,1.000000
1,convnextv2_pretrained_linear_probe,2,0.999107,0.998214,1.0,0.999107


## Reporting Note

Use the pretrained linear-probe rows for high-level ConvNeXt comparison if they are accepted by the group. Keep the `convnextv2_pretrained_finetune_suspect` row visible as a failed/suspect run rather than silently deleting it.